# Lab 1 — First MCP Server

This notebook maps the core MCP ideas to one working example: a `search_docs` tool that can be called from a LangGraph agent.

## Concept map

- What MCP solves
- host / client / server
- transport architecture
- stdio transport
- MCP Inspector
- `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`
- connect to a LangGraph agent

In [1]:
from pathlib import Path
import sys, os
ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
else:
    sys.path.insert(0, str(ROOT.parent))

from dotenv import load_dotenv
load_dotenv()

from src.tools.lab_helpers import concept_map, search_demo_corpus, demo_questions
from src.servers.search_server import search_docs
from src.agents.langgraph_agent import build_react_agent_with_tools
print('Imports ready')

C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports ready


In [2]:
concept_map()

[{'concept': 'What MCP solves',
  'summary': 'MCP standardizes how hosts, clients, and servers exchange context, tools, and prompts.'},
 {'concept': 'Transport architecture',
  'summary': 'The current Python SDK emphasizes stdio and Streamable HTTP; the docs also discuss legacy SSE examples for older ecosystems.'},
 {'concept': 'MCP server primitives',
  'summary': '@mcp.tool(), @mcp.resource(), and @mcp.prompt() expose callable tools, readable resources, and prompt templates.'},
 {'concept': 'LangGraph basics',
  'summary': 'StateGraph, conditional edges, compile(), and invoke() make graph control flow explicit and inspectable.'},
 {'concept': 'Human approval',
  'summary': 'interrupt() and interrupt_before pause a graph so a human can approve, edit, or reject an action.'},
 {'concept': 'Persistence',
  'summary': 'MemorySaver, SqliteSaver, and stores enable short-term and long-term memory across runs.'},
 {'concept': 'Multi-server agents',
  'summary': 'MultiServerMCPClient can expos

In [3]:
for q in demo_questions():
    print('\nQ:', q)
    print(search_docs(q, top_k=2)['rendered'])


Q: Explain what MCP solves and how hosts differ from servers.
1. What MCP solves (score=6): What MCP solves: MCP standardizes how hosts, clients, and servers exchange context, tools, and prompts.
2. MCP server primitives (score=2): MCP server primitives: @mcp.tool(), @mcp.resource(), and @mcp.prompt() expose callable tools, readable resources, and prompt templates.

Q: Show how a LangGraph StateGraph with conditional edges works.
1. LangGraph basics (score=4): LangGraph basics: StateGraph, conditional edges, compile(), and invoke() make graph control flow explicit and inspectable.
2. What MCP solves (score=1): What MCP solves: MCP standardizes how hosts, clients, and servers exchange context, tools, and prompts.

Q: How do I connect multiple MCP servers into one agent?
1. Multi-server agents (score=4): Multi-server agents: MultiServerMCPClient can expose tools from multiple MCP servers to one agent.
2. What MCP solves (score=3): What MCP solves: MCP standardizes how hosts, clients, an

## Run the MCP server separately

Open a terminal and run:

```bash
python -m src.servers.search_server
```

Then inspect it with MCP Inspector.

## Connect the same tool to a LangGraph agent

This cell uses Groq only when `GROQ_API_KEY` is available.

In [3]:
try:
    agent = build_react_agent_with_tools([search_docs])
    result = agent.invoke({'messages': [{'role': 'user', 'content': 'Explain what MCP solves in one paragraph.'}]})
    print(result)
except Exception as e:
    print('Agent not run:', e)

[06/19/26 14:41:30] INFO     HTTP Request: POST https://api.groq.com/openai/v1/chat/completions     ]8;id=949127;file://C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=213422;file://C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\httpx\_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

{'messages': [HumanMessage(content='Explain what MCP solves in one paragraph.', additional_kwargs={}, response_metadata={}, id='39aa98ae-2bb4-4b6b-8766-bf545dbcfd44'), AIMessage(content='The Minimum Cut Problem (MCP) is a fundamental problem in graph theory and computer science that involves finding the minimum cut in a flow network. A cut in a flow network is a partition of the vertices into two sets, and the capacity of the cut is the sum of the capacities of the edges that cross the partition. The Minimum Cut Problem solves the problem of finding the cut with the minimum capacity, which has numerous applications in various fields such as network reliability, transportation systems, and computer vision. By solving the MCP, one can identify the bottleneck or the weakest point in a network, which is essential for designing and optimizing network systems.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 262, 'total_tokens': 391, 'comp

## Edit points

Modify `src/tools/lab_helpers.py` to change the teaching corpus, and modify `src/servers/search_server.py` to change the MCP surface.